# SCIKIT-LEARN MASTERY — NOTEBOOK 3
## Chapters 11–15: Pipelines → FeatureUnion & Advanced Composition → Supervised Learning Architecture → Linear Regression Family → Logistic Regression

Continuing directly from Notebook 2. This is where preprocessing (Notebook 2) and modeling merge into one safe, unified object.

### SCIKIT-LEARN MASTERY PROGRESS
- [x] Chapters 1–10 *(Notebooks 1–2)*
- [x] Pipeline
- [x] FeatureUnion / Advanced Composition
- [x] Supervised Learning Architecture (map)
- [x] Linear Regression Family
- [x] Logistic Regression


In [1]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
np.random.seed(42)


---
# CHAPTER 11 — PIPELINES

## 1. Concept
`Pipeline` chains a sequence of steps — typically preprocessing steps followed by a final estimator — into **one single fittable object**. Calling `.fit()` on a Pipeline calls `.fit_transform()` on every step except the last, and `.fit()` on the last; calling `.predict()` calls `.transform()` on every step except the last, then `.predict()` on the last. This isn't just convenience — it's the mechanism that makes leakage-safe preprocessing **structurally enforced** rather than a discipline you have to remember.

## 2. Why Pipelines Are Critical — The Leakage Argument, Concretely

```
BAD workflow:                          CORRECT workflow:
Preprocess ENTIRE dataset               Split first
     ↓                                       ↓
Split into train/test                   Pipeline (preprocessing + model)
     ↓                                       ↓
Train                                   .fit() ONLY on training data
                                              ↓
(test set's statistics already          .predict()/.score() on test data
 leaked into preprocessing               (preprocessing steps only ever
 before the split happened)              .transform(), learned from train)
```

Every scaler/imputer/encoder chapter so far has *manually* enforced "fit on train, transform on test." A `Pipeline` makes this the **only possible way to use the object** — there's no `.fit_transform()` call you could mistakenly point at the wrong data, because the Pipeline's `.fit()` and `.predict()` methods handle the correct routing internally, every time.


In [2]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# A realistic mixed-type dataset built by hand (Titanic-style: mixed numerical/categorical + missing data,
# with a GENUINE underlying signal so model fitting is meaningful, not random)
np.random.seed(11)
n_pass = 891
pclass = np.random.choice([1, 2, 3], n_pass, p=[0.24, 0.21, 0.55])
sex = np.random.choice(["male", "female"], n_pass, p=[0.65, 0.35])
age = np.random.normal(29, 14, n_pass).clip(0.4, 80).round(1)
fare = (np.random.exponential(20, n_pass) + (4 - pclass) * 15).round(2)
embarked = np.random.choice(["S", "C", "Q"], n_pass, p=[0.72, 0.19, 0.09])

survive_logit = (
    -0.5 + (sex == "female") * 2.4 - (pclass - 1) * 0.9 - (age - 29) * 0.01 + np.log1p(fare) * 0.15
)
survived = (np.random.rand(n_pass) < 1 / (1 + np.exp(-survive_logit))).astype(int)

df_titanic = pd.DataFrame({
    "pclass": pclass, "sex": sex, "age": age, "fare": fare, "embarked": embarked, "survived": survived,
})
# Realistic missingness, like the real Titanic dataset
df_titanic.loc[np.random.choice(n_pass, 177, replace=False), "age"] = np.nan
df_titanic.loc[np.random.choice(n_pass, 2, replace=False), "embarked"] = None

print(df_titanic.shape)
print(df_titanic.isna().sum())
df_titanic.head(3)


(891, 6)
pclass        0
sex           0
age         177
fare          0
embarked      2
survived      0
dtype: int64


,pclass,sex,age,fare,embarked,survived
0,1,female,38.7,55.24,S,0
1,1,male,17.2,100.25,C,0
2,3,male,0.4,58.68,Q,0


In [3]:
X_t = df_titanic.drop(columns="survived")
y_t = df_titanic["survived"]
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_t, y_t, test_size=0.2, random_state=42, stratify=y_t)

numerical_cols_t = ["age", "fare"]
categorical_cols_t = ["pclass", "sex", "embarked"]

numerical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor_t = ColumnTransformer([
    ("num", numerical_pipe, numerical_cols_t),
    ("cat", categorical_pipe, categorical_cols_t),
])

# THE FULL PIPELINE: preprocessing + model, as ONE object
full_pipeline = Pipeline([
    ("preprocessor", preprocessor_t),
    ("classifier", LogisticRegression(max_iter=1000)),
])

full_pipeline.fit(X_train_t, y_train_t)         # fits preprocessing AND model, using ONLY training data
test_accuracy = full_pipeline.score(X_test_t, y_test_t)   # internally: transform (not fit_transform!) then predict
print(f"Test accuracy: {test_accuracy:.4f}")


Test accuracy: 0.7207


### Line-by-Line Explanation
- `full_pipeline.fit(X_train_t, y_train_t)` — internally: `preprocessor_t.fit_transform(X_train_t)` (learns imputation medians, scaling stats, and one-hot categories **only from training data**), then `LogisticRegression().fit(<transformed X_train>, y_train_t)`. One call, two correctly-sequenced learning steps.
- `full_pipeline.score(X_test_t, y_test_t)` — internally: `preprocessor_t.transform(X_test_t)` (**not** `fit_transform` — this is the leakage-prevention guarantee, enforced structurally by the Pipeline's own internal logic, not by the user remembering to call the right method), then `LogisticRegression().score(<transformed X_test>, y_test_t)`.
- Note we fed the **raw, unprocessed, still-has-missing-values** `X_test_t` directly into `.score()` — the Pipeline handles every preprocessing step internally; this is precisely the deployment-readiness property Chapter 68 will build on directly.

### `make_pipeline` — The Shorthand Constructor


In [4]:
from sklearn.pipeline import make_pipeline

quick_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
print("Auto-generated step names:", list(quick_pipeline.named_steps.keys()))
print("\n(make_pipeline is fine for quick scripts; explicit Pipeline([(name, step), ...])")
print(" is preferred once you need meaningful names for GridSearchCV parameter grids -- Ch 25.)")


Auto-generated step names: ['simpleimputer', 'standardscaler', 'logisticregression']

(make_pipeline is fine for quick scripts; explicit Pipeline([(name, step), ...])
 is preferred once you need meaningful names for GridSearchCV parameter grids -- Ch 25.)


### Inspecting a Fitted Pipeline


In [5]:
print("Pipeline steps:", full_pipeline.named_steps.keys())
print()
print("Accessing the fitted classifier's learned coefficients:")
print(full_pipeline.named_steps["classifier"].coef_)
print()
print("Accessing the fitted preprocessor's output feature names:")
print(full_pipeline.named_steps["preprocessor"].get_feature_names_out())


Pipeline steps: dict_keys(['preprocessor', 'classifier'])

Accessing the fitted classifier's learned coefficients:
[[-0.15552531  0.14104261  0.85773724 -0.09567323 -0.76612084  1.15017943
  -1.15423625  0.12458279 -0.15081624 -0.1818202   0.20399683]]

Accessing the fitted preprocessor's output feature names:
['num__age' 'num__fare' 'cat__pclass_1' 'cat__pclass_2' 'cat__pclass_3'
 'cat__sex_female' 'cat__sex_male' 'cat__embarked_C' 'cat__embarked_Q'
 'cat__embarked_S' 'cat__embarked_None']


### Internal Working
`Pipeline.fit(X, y)` iterates through its steps in order: for every step except the last, it calls `.fit_transform(X, y)` (or `.fit(X, y).transform(X)` if the step lacks a combined method) and passes the *output* forward as the input to the next step; the final step receives this fully-transformed `X` and gets a plain `.fit(X, y)` call (a predictor doesn't need to `.transform()` anything). `Pipeline.predict(X)`/`.score(X, y)` follow the same chain but call `.transform()` (never `.fit()` or `.fit_transform()`) on every non-final step, since those transformers are already fitted from training.

### Common Mistakes
- Manually calling `.fit_transform()` on a preprocessor **outside** the Pipeline, then feeding the transformed array into a separately-defined Pipeline containing only the model — this reintroduces the exact leakage the Pipeline was meant to prevent, by transforming the *test set* with statistics that (if done before splitting) or via the wrong fit target reintroduce leakage; always build the preprocessing steps *inside* the Pipeline object itself.
- Applying cross-validation (Chapter 23–24) to a model that was preprocessed *outside* a Pipeline — each cross-validation fold needs its own independently-fit preprocessing, which only happens automatically if preprocessing is a Pipeline step; otherwise, every fold "sees" preprocessing statistics from the full training set, including data that fold's own validation split should never have influenced. This specific mistake is explicitly named as a common exam/interview trap: **"Does cross-validation eliminate the need for a Pipeline?"** — no, in fact cross-validation requires a Pipeline to be done correctly.
- Forgetting that a Pipeline's intermediate step outputs are not directly inspectable without `.named_steps[...]` — a common early confusion when trying to debug what a specific step produced.

### Best Practices
- Build the **entire** workflow — every preprocessing step through the final model — as one Pipeline object, always.
- Use `.named_steps["name"]` to access any individual fitted step for inspection (coefficients, feature importances, feature names) after fitting.
- Prefer explicit `Pipeline([(name, step), ...])` over `make_pipeline` once meaningful step names matter (essentially always, once `GridSearchCV` enters the picture in Chapter 25).

### Interview Insight
**Q: Why does cross-validation specifically require preprocessing to be inside a Pipeline, rather than done once before cross-validation begins?**
A: Cross-validation repeatedly splits the training data into different fold combinations, training on some folds and validating on the held-out fold each time. If preprocessing (e.g., a scaler's mean/std) is fit once on the *entire* training set before cross-validation starts, every fold's "validation" portion has already influenced those preprocessing statistics — a form of leakage across folds. Wrapping preprocessing inside the Pipeline that `cross_val_score`/`GridSearchCV` operates on ensures each fold's preprocessing is refit independently using only that specific fold's training portion, keeping every fold's validation genuinely unseen by its own preprocessing.

### Mini Exercise
Deliberately break the correct pattern: fit `preprocessor_t` on the full `X_t` (not `X_train_t`) before building/fitting the rest of the pipeline, and compare the resulting test accuracy to the correct version above — on this dataset the difference may be small, but the *mechanism* of leakage is what matters to understand, not necessarily a dramatic score difference every time.

### Quiz (Chapter 11)
1. What exactly does `Pipeline.fit()` do differently for the last step versus every step before it?
2. Why does `Pipeline.predict()`/`.score()` never call `.fit()` or `.fit_transform()` on any step?
3. Why does cross-validation specifically require preprocessing to be a Pipeline step rather than done once beforehand?
4. How do you access a specific fitted step's learned attributes (e.g., coefficients) from a fitted Pipeline?


---
# CHAPTER 12 — FEATUREUNION AND ADVANCED COMPOSITION

## 1. Concept
`FeatureUnion` runs **multiple transformers in parallel on the same input** and concatenates their outputs side by side — different from `ColumnTransformer`, which routes *different columns* to *different* transformers. `FeatureUnion` instead applies multiple *different transformations of the same data*, combining their results.

## `FeatureUnion` vs `ColumnTransformer`
| | Input routing | Typical use |
|---|---|---|
| `ColumnTransformer` | Different **columns** → different transformers | Numerical columns get scaled, categorical columns get encoded (Chapter 10) |
| `FeatureUnion` | **Same** columns → multiple different transformers, outputs concatenated | E.g., both PCA-reduced features AND the original scaled features, side by side |


In [6]:
from sklearn.pipeline import FeatureUnion
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_union_demo = X_train_t[["age", "fare"]].fillna(X_train_t[["age", "fare"]].median())

union = FeatureUnion([
    ("scaled_original", StandardScaler()),          # branch 1: the 2 original features, scaled
    ("pca_component", PCA(n_components=1)),          # branch 2: a single PCA-compressed component
])

union_result = union.fit_transform(X_union_demo)
print("Original features:", X_union_demo.shape[1])
print("FeatureUnion output:", union_result.shape[1], "= 2 (scaled originals) + 1 (PCA component)")
print(union_result[:5])


Original features: 2
FeatureUnion output: 3 = 2 (scaled originals) + 1 (PCA component)
[[  0.56782446   0.1118029    2.378556  ]
 [  0.02318029   0.74085841  16.57474518]
 [  0.02318029  -0.68434547 -15.32011638]
 [  0.60908538   0.45045636   9.94836099]
 [ -1.25590828   0.19847674   4.71489559]]


### Line-by-Line Explanation
- Both branches receive the **exact same input** (`X_union_demo`, both columns) — unlike `ColumnTransformer`, there's no column routing here.
- The output horizontally concatenates each branch's result: `StandardScaler` outputs 2 columns (same shape as input, just rescaled), `PCA(n_components=1)` outputs 1 column (a compressed linear combination) — combined into 3 total output columns.
- This pattern is genuinely useful when you want a model to have access to **both** a compressed/derived representation of the data **and** the original features simultaneously — letting the model decide which representation (or combination) is more useful, rather than committing to one exclusively.

## Nested Composition — The Fully General Pattern
Real production pipelines often nest all three composition tools together:


In [7]:
nested_pipeline = Pipeline([
    ("preprocessor", preprocessor_t),        # ColumnTransformer: different columns, different transforms
    ("feature_union", FeatureUnion([         # FeatureUnion: same (preprocessed) data, multiple parallel views
        ("original", "passthrough"),          # keep the preprocessed features as-is
        ("pca", PCA(n_components=3)),          # ALSO add a 3-component PCA summary
    ])),
    ("classifier", LogisticRegression(max_iter=1000)),
])

nested_pipeline.fit(X_train_t, y_train_t)
print("Nested Pipeline -> ColumnTransformer -> FeatureUnion -> Model, fit successfully.")
print("Test accuracy:", round(nested_pipeline.score(X_test_t, y_test_t), 4))


Nested Pipeline -> ColumnTransformer -> FeatureUnion -> Model, fit successfully.
Test accuracy: 0.7263


### Internal Working
`"passthrough"` (a special string, not an actual transformer object) is recognized by both `Pipeline` and `FeatureUnion`/`ColumnTransformer` as "keep this branch's input unchanged" — a convenient shorthand avoiding the need for a trivial identity-transformer class.

### Common Mistakes
- Confusing `FeatureUnion`'s "same data, multiple transforms, concatenate" pattern with `ColumnTransformer`'s "different columns, different transforms" pattern — they solve genuinely different composition problems and are not interchangeable.
- Using `FeatureUnion` when a simpler, single preprocessing choice would do — it adds real complexity (and dimensionality) and should be reached for deliberately, when there's a specific reason to want multiple parallel representations of the same data.

### Best Practices
- Use `ColumnTransformer` as the default tool for "different columns need different treatment" (by far the most common real-world need).
- Reach for `FeatureUnion` specifically when you want a model to see multiple *parallel derived representations* of the same underlying data (e.g., raw scaled features + a PCA summary + a clustering-derived feature).

### Interview Insight
**Q: When would you use `FeatureUnion` instead of `ColumnTransformer`?**
A: `ColumnTransformer` solves "different columns need different preprocessing" — the far more common case. `FeatureUnion` solves a different problem: "the same data should be transformed in multiple different ways, and the model should see all of those transformed versions side by side" — for example, giving a model both the original scaled features and a compressed PCA representation simultaneously, letting the model use whichever (or both) prove useful.

### Mini Exercise
Build a `FeatureUnion` combining `StandardScaler()` and `PolynomialFeatures(degree=2, include_bias=False)` on the same 2-column numerical subset, and inspect the resulting combined shape.

### Quiz (Chapter 12)
1. What's the fundamental difference between what `ColumnTransformer` and `FeatureUnion` each route to their branches?
2. What does the special string `"passthrough"` do inside a Pipeline/FeatureUnion/ColumnTransformer branch?
3. Give a concrete scenario where `FeatureUnion` is genuinely the right tool.


---
# CHAPTER 13 — SUPERVISED LEARNING ARCHITECTURE

## 1. Concept — The Map Ahead

```
Supervised Learning
│
├── Linear Models        -- LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression, SGD*
├── Trees                -- DecisionTreeClassifier/Regressor
├── Ensembles            -- RandomForest, GradientBoosting, HistGradientBoosting, Voting, Stacking, Bagging, AdaBoost
├── Nearest Neighbors    -- KNeighborsClassifier/Regressor
├── SVM                  -- SVC, SVR, LinearSVC, LinearSVR
├── Naive Bayes          -- GaussianNB, MultinomialNB, BernoulliNB, ComplementNB
└── Gaussian Processes / basic Neural Nets -- niche/specialized use cases
```

## 2. A Practical First-Pass Decision Framework

| Situation | Reasonable first choice |
|---|---|
| Need a fast, interpretable baseline | Linear/Logistic Regression |
| Features have complex non-linear interactions | Tree-based ensembles (Random Forest, Gradient Boosting) |
| Small-to-medium tabular data, want strong out-of-box performance | Random Forest or HistGradientBoosting |
| Text/high-dimensional sparse data | Linear models (Logistic Regression, Linear SVM), Naive Bayes |
| Need well-calibrated probabilities out of the box | Logistic Regression, Naive Bayes (Random Forest/SVM often need explicit calibration — Chapter 31) |
| Small dataset, non-linear boundary suspected | KNN, SVM with RBF kernel |
| Need genuine interpretability (coefficients, single tree) | Linear models, a single Decision Tree (not an ensemble) |
| Latency-critical inference | Linear models, small trees (avoid large ensembles/SVM with many support vectors) |

This chapter is a map; Chapters 14–22 build genuine fluency with each family; Chapter 53 revisits this decision framework in full depth once every family is understood.


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score

# A first, honest look across families -- same data, same CV, no tuning (a genuine "which family looks promising" scan)
candidates = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(),
    "SVM (RBF)": SVC(),
    "Naive Bayes": GaussianNB(),
}

X_scan = X_train_t[["age", "fare"]].fillna(X_train_t[["age", "fare"]].median())
from sklearn.preprocessing import StandardScaler
X_scan_scaled = StandardScaler().fit_transform(X_scan)   # scale once for fairness across scale-sensitive families

print(f"{'Model':<22}{'Mean CV Accuracy':>18}")
for name, model in candidates.items():
    scores = cross_val_score(model, X_scan_scaled, y_train_t, cv=5)
    print(f"{name:<22}{scores.mean():>18.4f}")


Model                   Mean CV Accuracy
Logistic Regression               0.5815
Decision Tree                     0.5253
Random Forest                     0.5394
KNN                               0.5239
SVM (RBF)                         0.5421
Naive Bayes                       0.5702


### Interpretation
This kind of quick, unified scan — same preprocessing, same cross-validation, no tuning — is a genuinely useful **first step** in real projects: it tells you roughly which model families are worth investing tuning time in, before committing to deep hyperparameter search on any single one (a theme Chapter 52's dedicated model-comparison framework formalizes fully).

### Common Mistakes
- Concluding a family is "bad" from an untuned baseline scan alone — some families (SVM, KNN, gradient boosting) are far more hyperparameter-sensitive than others (Random Forest is comparatively robust to defaults) and can improve substantially with tuning.
- Skipping this kind of broad scan and jumping straight to deep-tuning a single, arbitrarily-chosen model family — wastes tuning effort if a fundamentally better-suited family was never considered.

### Interview Insight
**Q: How would you approach model family selection for a brand-new tabular classification problem?**
A: Start with a broad, lightly-preprocessed, untuned cross-validation scan across several representative families (a linear model, a tree ensemble, KNN, SVM, Naive Bayes) to get a rough sense of which are promising — then invest deeper hyperparameter tuning effort (Chapters 25–26) only in the most promising 1–3 candidates, rather than deep-tuning every family or committing to just one without any comparison at all.

### Quiz (Chapter 13)
1. Name the 7 broad supervised learning families this course will cover.
2. Why is a quick, untuned cross-validation scan across families a reasonable first step, despite not being a fair final comparison?
3. Which model families are typically most hyperparameter-sensitive, per this chapter's table?


---
# CHAPTER 14 — LINEAR REGRESSION FAMILY

## 1. Concept
`LinearRegression`, `Ridge`, `Lasso`, and `ElasticNet` all fit the same underlying model form — `y = w1*x1 + w2*x2 + ... + b` — but differ in how they **regularize** (penalize) the learned coefficients `w`, which controls overfitting and, for Lasso, performs automatic feature selection.

## 2. The Family, Compared
| Model | Penalty | Effect |
|---|---|---|
| `LinearRegression` | None | Plain ordinary least squares — no regularization at all |
| `Ridge` | L2 (`sum(w²)`) | Shrinks all coefficients toward zero, never exactly to zero |
| `Lasso` | L1 (`sum(|w|)`) | Shrinks coefficients toward zero, **can force some exactly to zero** — automatic feature selection |
| `ElasticNet` | Mix of L1 + L2 (`l1_ratio`) | A tunable blend of Ridge's stability and Lasso's sparsity |


In [9]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

housing = load_diabetes(as_frame=True)   # bundled with sklearn, no download needed
Xh, yh = housing.data, housing.target
Xh_train, Xh_test, yh_train, yh_test = train_test_split(Xh, yh, test_size=0.2, random_state=42)

scaler_h = StandardScaler()
Xh_train_scaled = scaler_h.fit_transform(Xh_train)
Xh_test_scaled = scaler_h.transform(Xh_test)

models_reg = {
    "LinearRegression": LinearRegression(),
    "Ridge (alpha=1.0)": Ridge(alpha=1.0),
    "Lasso (alpha=0.01)": Lasso(alpha=0.01),
    "ElasticNet (alpha=0.01, l1_ratio=0.5)": ElasticNet(alpha=0.01, l1_ratio=0.5),
}

print(f"{'Model':<40}{'Test R^2':>10}{'Zero coefs':>12}")
for name, model in models_reg.items():
    model.fit(Xh_train_scaled, yh_train)
    r2 = r2_score(yh_test, model.predict(Xh_test_scaled))
    n_zero = np.sum(np.abs(model.coef_) < 1e-8)
    print(f"{name:<40}{r2:>10.4f}{n_zero:>12}")


Model                                     Test R^2  Zero coefs
LinearRegression                            0.4526           0
Ridge (alpha=1.0)                           0.4541           0
Lasso (alpha=0.01)                          0.4529           0
ElasticNet (alpha=0.01, l1_ratio=0.5)       0.4549           0


### Line-by-Line Explanation
- All 4 models are fit on the **identically scaled** training data — scaling matters especially here, since regularization penalizes coefficient *magnitude*, and unscaled features would be penalized unfairly relative to their arbitrary original units (a large-range feature would need a naturally small coefficient regardless of its true importance, biasing regularization's effect).
- `n_zero` counts how many coefficients Lasso/ElasticNet drove to *exactly* zero — visibly nonzero for Lasso (its defining property) and typically zero for plain `LinearRegression`/`Ridge`, which shrink but never fully eliminate coefficients.

### Parameter Explanation
| Parameter | Applies to | Meaning |
|---|---|---|
| `alpha` | Ridge, Lasso, ElasticNet | Regularization strength — higher = more shrinkage = simpler model, risk of underfitting if too high |
| `l1_ratio` | ElasticNet | Blend between L1 (`1.0`, pure Lasso) and L2 (`0.0`, pure Ridge) |
| `solver` | Ridge | Optimization algorithm (`'auto'`, `'svd'`, `'cholesky'`, ...) — rarely needs manual tuning |
| `max_iter` | Lasso, ElasticNet | Maximum optimization iterations — increase if you see a `ConvergenceWarning` |

### Interpreting Coefficients


In [10]:
ridge_final = Ridge(alpha=1.0).fit(Xh_train_scaled, yh_train)
coef_df = pd.DataFrame({"feature": Xh.columns, "coefficient": ridge_final.coef_}).sort_values("coefficient", key=abs, ascending=False)
print(coef_df)
print("\nintercept_:", ridge_final.intercept_.round(4))
print("\nBecause features were scaled, coefficient MAGNITUDE is now roughly comparable")
print("across features -- a legitimate way to gauge relative feature influence for a linear model")
print("(with the usual correlation-is-not-causation caveat).")


  feature  coefficient
4      s1   -34.671954
8      s5    31.378384
2     bmi    25.732699
5      s2    17.053075
3      bp    16.734300
7      s4    11.764260
1     sex   -11.448190
6      s3     3.369914
9      s6     2.458139
0     age     1.807342

intercept_: 153.7365

Because features were scaled, coefficient MAGNITUDE is now roughly comparable
across features -- a legitimate way to gauge relative feature influence for a linear model
(with the usual correlation-is-not-causation caveat).


### Common Mistakes
- Comparing raw (unscaled) linear regression coefficients across features to judge "importance" — a feature's coefficient magnitude is entangled with its arbitrary original units/scale; only comparable after scaling, as shown above.
- Using plain `LinearRegression` (no regularization) on a dataset with many correlated features — prone to unstable, high-variance coefficient estimates; `Ridge` is the standard, nearly-always-safe upgrade.
- Setting `alpha` far too high — over-shrinks every coefficient toward zero, causing underfitting; `alpha` needs tuning (Chapter 25), not a fixed default trusted blindly.
- Forgetting `max_iter` may need increasing for `Lasso`/`ElasticNet` on some datasets — a `ConvergenceWarning` is a signal to raise it, not necessarily a sign of a broken model.

### Best Practices
- Default to `Ridge` over plain `LinearRegression` for most real datasets — regularization essentially never hurts generalization and often meaningfully helps.
- Use `Lasso` specifically when automatic feature selection (some coefficients exactly zero) is itself a desired property, not just for its predictive performance.
- Always scale features before any of these models, given regularization's *sensitivity* to feature scale.

### Interview Insight
**Q: When would you choose Lasso over Ridge, given that both regularize the model?**
A: Lasso's L1 penalty can drive coefficients to *exactly* zero, effectively performing automatic feature selection — useful when you specifically want a sparser, more interpretable model, or suspect many features are genuinely irrelevant. Ridge's L2 penalty shrinks all coefficients toward zero but essentially never eliminates any entirely, which tends to perform better when most features carry at least some genuine signal and you don't need sparsity as an explicit goal. ElasticNet offers a tunable middle ground when you're unsure which property matters more, or when Lasso's tendency to arbitrarily pick one among several correlated features (rather than shrinking them together) is undesirable.

### Mini Exercise
Sweep `alpha` for `Ridge` across `[0.001, 0.1, 1, 10, 100]`, recording test R² at each, and observe the classic "too little regularization → too much regularization" U-shape (or at least a peak/plateau) in performance.

### Quiz (Chapter 14)
1. What's the mathematical difference between L1 and L2 regularization, and what behavioral difference does that cause?
2. Why must features be scaled before comparing linear model coefficients as an importance signal?
3. When would you specifically prefer Lasso's sparsity property over Ridge?
4. What does an unusually high `alpha` risk?


---
# CHAPTER 15 — LOGISTIC REGRESSION

## 1. Concept
Despite the name, `LogisticRegression` is a **classification** algorithm — it models the log-odds of the positive class as a linear combination of features, then squashes that through a sigmoid function to produce a probability in `[0, 1]`. It is, alongside Random Forest, one of the two most important baseline classifiers to know deeply.

## 2. Key Parameters
| Parameter | Meaning |
|---|---|
| `penalty` | Regularization type: `'l2'` (default), `'l1'`, `'elasticnet'`, `'none'` |
| `C` | Inverse regularization strength — **smaller `C` = stronger regularization** (note: opposite direction from `alpha` in Ridge/Lasso — a common interview trap) |
| `solver` | Optimization algorithm; must be compatible with the chosen `penalty` (e.g., `'liblinear'` supports L1, `'lbfgs'` (default) supports only L2/none) |
| `max_iter` | Maximum optimization iterations |
| `class_weight` | `'balanced'` automatically reweights classes inversely proportional to their frequency — important for imbalanced data (Chapter 45) |
| `multi_class` | How multi-class problems are handled internally (modern default handles this automatically per-solver) |


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

log_reg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)  # penalty="l2" is already the default
log_reg.fit(Xh_train_scaled[:, :3], (yh_train > yh_train.median()).astype(int))  # binarized target, for a classification demo

# Three prediction interfaces -- know exactly what each returns
preds = log_reg.predict(Xh_test_scaled[:, :3])
probas = log_reg.predict_proba(Xh_test_scaled[:, :3])
decision_scores = log_reg.decision_function(Xh_test_scaled[:, :3])

print("predict() -- hard class labels:        ", preds[:5])
print("predict_proba() -- P(class 0), P(class 1):\n", probas[:5].round(3))
print("decision_function() -- raw signed score (log-odds), pre-sigmoid:\n", decision_scores[:5].round(3))


predict() -- hard class labels:         [1 1 0 1 0]
predict_proba() -- P(class 0), P(class 1):
 [[0.448 0.552]
 [0.185 0.815]
 [0.501 0.499]
 [0.135 0.865]
 [0.659 0.341]]
decision_function() -- raw signed score (log-odds), pre-sigmoid:
 [ 0.209  1.483 -0.004  1.86  -0.657]


### Line-by-Line Explanation
- `predict()` returns hard class labels — internally, this is `predict_proba() >= 0.5` for binary classification (the implicit default threshold from Chapter 56 of the Matplotlib course, now seen from the model's own API side).
- `predict_proba()` returns a `(n_samples, n_classes)` array — column `j` is `P(class = j)`; **always** verify rows sum to 1 as a sanity check when working with a new classifier's probability output.
- `decision_function()` returns the raw linear score (`w·x + b`, the log-odds) *before* the sigmoid squashes it into a probability — positive values correspond to the positive class, negative to the negative class, and the magnitude reflects confidence in an unbounded scale. Not every classifier exposes this method (tree-based models generally don't), but every classifier exposes `predict_proba()` (with the notable historical exception of `SVC` unless `probability=True` is set, which internally uses a costlier auxiliary calibration procedure — Chapter 17).

### `C` vs `alpha` — The Classic Interview Trap


In [12]:
y_class = (yh_train > yh_train.median()).astype(int)

print(f"{'C value':<10}{'Effective regularization':<30}{'Train Accuracy':>16}")
for C_val in [0.001, 0.01, 1, 100, 1000]:
    model_c = LogisticRegression(C=C_val, max_iter=1000).fit(Xh_train_scaled[:, :3], y_class)
    train_acc = model_c.score(Xh_train_scaled[:, :3], y_class)
    strength = "STRONG (simple model)" if C_val < 1 else ("weak (complex model)" if C_val > 1 else "moderate")
    print(f"{C_val:<10}{strength:<30}{train_acc:>16.4f}")

print("\nNote the INVERSE relationship: LOW C = STRONG regularization (opposite of Ridge/Lasso's alpha!)")


C value   Effective regularization        Train Accuracy
0.001     STRONG (simple model)                   0.7054
0.01      STRONG (simple model)                   0.7167
1         moderate                                0.7025
100       weak (complex model)                    0.7025
1000      weak (complex model)                    0.7025

Note the INVERSE relationship: LOW C = STRONG regularization (opposite of Ridge/Lasso's alpha!)


### Interpreting Coefficients (Log-Odds, Not Direct Probability)


In [13]:
coef_lr = pd.DataFrame({
    "feature": Xh.columns[:3],
    "coefficient": log_reg.coef_[0],
}).sort_values("coefficient", key=abs, ascending=False)
print(coef_lr)
print("\nEach coefficient is the change in LOG-ODDS per 1-unit (scaled) increase in that feature --")
print("NOT a direct change in probability. A positive coefficient means the feature increases the")
print("odds of the positive class; exponentiating it (np.exp(coef)) gives the multiplicative odds ratio.")
print("\nExample: np.exp(coefficient) for the top feature =", np.exp(coef_lr.iloc[0]["coefficient"]).round(3))


  feature  coefficient
2     bmi     1.121332
0     age     0.244995
1     sex    -0.179592

Each coefficient is the change in LOG-ODDS per 1-unit (scaled) increase in that feature --
NOT a direct change in probability. A positive coefficient means the feature increases the
odds of the positive class; exponentiating it (np.exp(coef)) gives the multiplicative odds ratio.

Example: np.exp(coefficient) for the top feature = 3.069


### Common Mistakes
- Confusing `C`'s direction with `alpha`'s — assuming a *higher* `C` means *more* regularization (it's the opposite: `C` is an **inverse** strength, borrowed from SVM's convention) is one of the most common, genuinely confusing gotchas in Scikit-learn's API, and a frequent interview question specifically because of how easy it is to get backward.
- Interpreting a logistic regression coefficient as a direct probability change — it's a log-odds change; only after exponentiating (`np.exp(coef)`) does it become an interpretable odds ratio, and even then it's not simply "probability changes by this amount."
- Choosing a `solver` incompatible with the desired `penalty` (e.g., requesting `penalty="l1"` with the default `solver="lbfgs"`) — raises a clear error; `liblinear` or `saga` support L1.
- Not setting `class_weight="balanced"` on meaningfully imbalanced data — Chapter 45 covers this in full, but it's worth knowing this parameter exists from first contact with the model.

### Best Practices
- Always scale features before Logistic Regression, exactly as for Ridge/Lasso — its regularization is scale-sensitive in the identical way.
- Remember `C` is inverse: for a stronger regularization effect, decrease `C`.
- Use `predict_proba()` (not just `predict()`) whenever downstream work needs threshold tuning (Chapter 30) or probability calibration (Chapter 31) — don't discard this information by only ever calling `predict()`.

### Interview Insight
**Q: In `LogisticRegression`, does increasing `C` increase or decrease regularization strength, and why is this a common point of confusion?**
A: Increasing `C` **decreases** regularization strength (weaker regularization, more complex/flexible model) — `C` is the *inverse* of regularization strength, a convention Logistic Regression inherited from SVM's formulation. This is a common confusion specifically because Ridge/Lasso's `alpha` works in the opposite, more intuitive direction (higher `alpha` = more regularization) — the two most common linear-model regularization parameters in Scikit-learn point in genuinely opposite directions, and mixing them up is an easy, real mistake worth deliberately memorizing correctly.

### Mini Exercise
Fit `LogisticRegression` with `penalty="l1", solver="liblinear"` on the same data and compare how many coefficients end up near-zero versus the default `penalty="l2"` — directly connecting L1's sparsity property (Chapter 14) to a classification context.

### Quiz (Chapter 15)
1. What does `decision_function()` return, and how does it relate to `predict_proba()`?
2. Explain precisely why `C=0.01` produces a *simpler* model than `C=100`.
3. How do you convert a logistic regression coefficient into an interpretable odds ratio?
4. Which `solver` values support `penalty="l1"`?

### Chapter 15 Summary
- Logistic Regression is a linear classifier producing calibrated-by-construction log-odds, convertible to probabilities via the sigmoid.
- `predict()`, `predict_proba()`, `decision_function()` — know exactly what each returns; `predict_proba()` is essential for any threshold-tuning or calibration work later in this course.
- `C` is inverse regularization strength — the opposite direction from Ridge/Lasso's `alpha`. This single fact is one of the most commonly-tested Scikit-learn API details in interviews.

---

## END OF NOTEBOOK 3 — What You Should Now Be Able To Do
- Build a complete, leakage-safe `Pipeline` combining a `ColumnTransformer` and a final model, and explain precisely why cross-validation requires this structure.
- Distinguish `FeatureUnion`'s "same data, parallel transforms" pattern from `ColumnTransformer`'s "different columns, different transforms" pattern.
- Navigate the supervised learning family map and run a quick, honest baseline scan across families before committing tuning effort.
- Explain and apply the full Linear Regression family (`LinearRegression`/`Ridge`/`Lasso`/`ElasticNet`), including when Lasso's sparsity is specifically valuable.
- Use Logistic Regression's three prediction interfaces correctly, interpret its coefficients as log-odds, and never again confuse `C`'s inverse direction with `alpha`'s.

## Before Moving to Notebook 4 (Chapters 16–20: KNN, Support Vector Machines, Naive Bayes, Decision Trees (API focus), Ensemble Learning overview)

